<a href="https://colab.research.google.com/github/2001lida/PythonLession2/blob/hw_5/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B55.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl

df = pl.read_csv("train.csv")
print("Размер датасета (строки, столбцы):")
print(df.shape)

print("\nСхема датафрейма (имя столбца → тип данных):")
print(df.schema)

print("\nТипы данных (dtypes):")
print(df.dtypes)

# ------------------------------------------------------------------
# 3. Пропущенные значения
# ------------------------------------------------------------------
print("\nКоличество пропусков (null) по столбцам:")
print(df.null_count())

# ------------------------------------------------------------------
# 4. Описательные статистики
# ------------------------------------------------------------------
print("\nОписательные статистики (describe):")
print(df.describe())

# ------------------------------------------------------------------
# 5. Средние значения числовых столбцов
# ------------------------------------------------------------------
print("\nСредние значения всех числовых столбцов:")
print(
    df.select(pl.selectors.numeric().mean())
)

# ------------------------------------------------------------------
# 6. Дополнительная информация
# ------------------------------------------------------------------

# Распределение целевой переменной
print("\nРаспределение Survived:")
print(
    df.select(pl.col("Survived").value_counts())
)

# Уникальные значения категориальных признаков
print("\nУникальные значения Sex и Embarked:")

print("Распределение Sex:")
print(df.select(pl.col("Sex").value_counts()))

print("\nРаспределение Embarked:")
print(df.select(pl.col("Embarked").value_counts()))

#3
# Получаем столбец Pclass
pclass = df.get_column("Pclass")

# Считаем количество пассажиров каждого класса
pclass_counts = pclass.value_counts()

print(pclass_counts)

#4
# Группировка по полу и подсчёт выживших
survived_by_sex = (df.group_by("Sex").agg(pl.col("Survived").sum().alias("survived_count")))

print(survived_by_sex)

#5
# Фильтрация пассажиров старше 44 лет
older_than_44 = df.filter(
    pl.col("Age") > 44
)

print(older_than_44)

Размер датасета (строки, столбцы):
(891, 12)

Схема датафрейма (имя столбца → тип данных):
Schema({'PassengerId': Int64, 'Survived': Int64, 'Pclass': Int64, 'Name': String, 'Sex': String, 'Age': Float64, 'SibSp': Int64, 'Parch': Int64, 'Ticket': String, 'Fare': Float64, 'Cabin': String, 'Embarked': String})

Типы данных (dtypes):
[Int64, Int64, Int64, String, String, Float64, Int64, Int64, String, Float64, String, String]

Количество пропусков (null) по столбцам:
shape: (1, 12)
┌─────────────┬──────────┬────────┬──────┬───┬────────┬──────┬───────┬──────────┐
│ PassengerId ┆ Survived ┆ Pclass ┆ Name ┆ … ┆ Ticket ┆ Fare ┆ Cabin ┆ Embarked │
│ ---         ┆ ---      ┆ ---    ┆ ---  ┆   ┆ ---    ┆ ---  ┆ ---   ┆ ---      │
│ u32         ┆ u32      ┆ u32    ┆ u32  ┆   ┆ u32    ┆ u32  ┆ u32   ┆ u32      │
╞═════════════╪══════════╪════════╪══════╪═══╪════════╪══════╪═══════╪══════════╡
│ 0           ┆ 0        ┆ 0      ┆ 0    ┆ … ┆ 0      ┆ 0    ┆ 687   ┆ 2        │
└─────────────┴──────────

In [ ]:
import pandas as pd
import bottleneck as bn

#1
df = pd.read_csv("train.csv")

#2
ages = df["Age"].values

mean_age = bn.nanmean(ages)
std_age = bn.nanstd(ages)

print("Средний возраст:", mean_age)
print("Стандартное отклонение возраста:", std_age)

#3
df["Fare_new"] = [
    row.Fare * 1.3 if pd.notna(row.Fare) else None
    for row in df.itertuples()
]

print(df[["Fare", "Fare_new"]].head())

Средний возраст: 29.69911764705882
Стандартное отклонение возраста: 14.516321150817317
      Fare  Fare_new
0   7.2500   9.42500
1  71.2833  92.66829
2   7.9250  10.30250
3  53.1000  69.03000
4   8.0500  10.46500


In [ ]:
import pandas as pd

df = pd.read_csv("Housing.csv")

memory_before = df.memory_usage(deep=True).sum()
print(f"Потребление памяти до оптимизации: {memory_before / 1024:.2f} KB")

print(df.dtypes)

optimized_df = df.copy()

# Числовые столбцы
optimized_df["price"] = optimized_df["price"].astype("int32")
optimized_df["area"] = optimized_df["area"].astype("int32")

optimized_df["bedrooms"] = optimized_df["bedrooms"].astype("int8")
optimized_df["bathrooms"] = optimized_df["bathrooms"].astype("int8")
optimized_df["stories"] = optimized_df["stories"].astype("int8")
optimized_df["parking"] = optimized_df["parking"].astype("int8")

# Категориальные столбцы
categorical_columns = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea",
    "furnishingstatus",
]

# price (int64)
# Цена дома — целое число, диапазон обычно < 2^31
# → int32 достаточно

# area (int64)
# Площадь дома — целое число, обычно < 10000
# → int32 достаточно

# bedrooms, bathrooms, stories, parking (int64)
# Малые целые числа (обычно 0–10)
# → int8 достаточно (экономия памяти в 8 раз)

# mainroad, guestroom, basement, hotwaterheating,
# airconditioning, prefarea, furnishingstatus (object)
# Категориальные признаки с малым числом уникальных значений
# → category (максимальная экономия памяти)

for col in categorical_columns:
    optimized_df[col] = optimized_df[col].astype("category")

memory_after = optimized_df.memory_usage(deep=True).sum()
print(f"Память после оптимизации: {memory_after / 1024:.2f} KB")

reduction = (memory_before - memory_after) / memory_before * 100
print(f"Снижение потребления памяти: {reduction:.2f}%")

Потребление памяти до оптимизации: 221.92 KB
price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object
Память после оптимизации: 11.76 KB
Снижение потребления памяти: 94.70%
